In [27]:
for model in gemini_client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

CHATBOT EVALUATION

In [24]:
import os 
from dotenv import load_dotenv 
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [35]:
##create data points 
from langsmith import Client 
client = Client()

dataset_name = "cchatbot's evalauation"
dataset= client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        }
    ]
)


{'example_ids': ['e389a35b-2dba-4fee-8092-a797a5dc5945',
  'e70b8b6d-5058-46bb-b780-321d65094e79'],
 'count': 2,
 'as_of': '2026-03-12T10:58:30.492347177Z'}

In [ ]:
## Define Metrics (LLM as a judge)

import os
from dotenv import load_dotenv
from google import genai

# Load environment variables
load_dotenv()

# Get API key from .env
api_key = os.getenv("GOOGLE_API_KEY")

# Initialize Gemini client
gemini_client = genai.Client(api_key=api_key)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:

    user_content = f"""
    You are grading the following question:
    {inputs['question']}

    Here is the real answer:
    {reference_outputs['answer']}

    You are grading the following predicted answer:
    {outputs['response']}

    Respond with CORRECT or INCORRECT.

    Grade:
    """

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_content
    )

    result = response.text.strip()

    return result == "CORRECT"

In [ ]:
# Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

RUN THE EVALUATION

In [38]:
default_instructions = "Respond to the user's question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "gemini-2.5-flash", instructions: str = default_instructions) -> str:
    
    prompt = f"""
    {instructions}

    Question: {question}
    """

    response = gemini_client.models.generate_content(
            model=model,
            contents=prompt
        )

    return response.text.strip()

In [39]:
def ls_target(inputs: dict):
    return {"response": my_app(inputs["question"])}

In [40]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-2.5-flash-chatbot",
    max_concurrency=1
)

View the evaluation results for experiment: 'gemini-2.5-flash-chatbot-3d7783f2' at:
https://smith.langchain.com/o/df5c8294-6cf0-4815-9067-4d0ffa7d80fb/datasets/59aba2a1-a45e-4848-bc81-57b12feb8e3a/compare?selectedSessions=71e5f091-db05-42ad-9228-20f0a61efc4a




2it [00:06,  3.25s/it]


In [49]:
def ls_target(inputs: dict):
    return {"response": my_app(inputs["question"], model="gemini-2.5-flash-lite")}

In [50]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-2.5-flash-lite",
    max_concurrency=1
)

View the evaluation results for experiment: 'gemini-2.5-flash-lite-878f65fe' at:
https://smith.langchain.com/o/df5c8294-6cf0-4815-9067-4d0ffa7d80fb/datasets/59aba2a1-a45e-4848-bc81-57b12feb8e3a/compare?selectedSessions=5afbc8c5-7b12-4367-84e3-0dd3850a3220




2it [00:03,  1.93s/it]


EVALUATION FOR RAG

In [54]:
!pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [61]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# URLs
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/"
]

# Load documents
docs = []

for url in urls:
    loader = WebBaseLoader(
        url,
        requests_kwargs={
            "headers": {"User-Agent": "Mozilla/5.0"},
            "timeout": 10
        }
    )
    docs.extend(loader.load())

# Split documents
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=550,
    chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs)

# Gemini embeddings (CORRECT MODEL)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# Vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [62]:
retriever.invoke("what are agents?")

[Document(id='e2d92aa0-b19f-4912-90f2-8f939040f0e6', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [67]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)

llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='models/gemini-2.5-flash', temperature=0.0, client=<google.genai.client.Client object at 0x000001AB595F4110>, default_metadata=(), model_kwargs={})

In [68]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [69]:
rag_bot("What is agents")

{'answer': 'In the context of the provided document, an agent is a system that uses a Large Language Model (LLM) as its core controller, essentially functioning as its brain. These LLM-powered autonomous agents are complemented by key components such as planning, memory, and tool use. They are designed to be powerful general problem solvers, capable of breaking down tasks, reflecting on past actions, retaining information, and utilizing external APIs.',
 'documents': [Document(id='6f0ecc6b-01fc-46bd-aba3-54ab84007b66', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful gene

In [70]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['87560216-3df0-4519-8a08-a169c067d89b'],
 'count': 1,
 'as_of': '2026-03-13T10:18:30.30413869Z'}

In [71]:
from typing_extensions import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Correctness Output Schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


# Correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step-by-step before giving the final decision.
"""


# Gemini LLM with structured output
grader_llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
).with_structured_output(CorrectnessGrade)


# Evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """Evaluator for RAG answer accuracy"""

    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return grade["correct"]

In [72]:
from typing_extensions import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os




# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide whether the answer addresses the question"]


# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step-by-step before giving the final decision.
"""


# Gemini grader LLM
relevance_llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
).with_structured_output(RelevanceGrade)


# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """Evaluator for RAG answer relevance"""

    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
"""

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

In [73]:
from typing_extensions import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "True if the answer is grounded in the documents"]


# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain hallucinated information outside the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning step-by-step before giving the final decision.
"""


# Gemini Grader LLM
grounded_llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
).with_structured_output(GroundedGrade)


# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """Evaluator for RAG answer groundedness"""

    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])

    answer = f"""
FACTS:
{doc_string}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])

    return grade["grounded"]

In [74]:
from typing_extensions import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os


# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]


# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning step-by-step before giving the final decision.
"""


# Gemini Grader LLM
retrieval_relevance_llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
).with_structured_output(RetrievalRelevanceGrade)


# Evaluator
def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """Evaluator for retrieved document relevance"""

    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])

    answer = f"""
FACTS:
{doc_string}

QUESTION:
{inputs['question']}
"""

    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

In [76]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-aa3cb5d2' at:
https://smith.langchain.com/o/df5c8294-6cf0-4815-9067-4d0ffa7d80fb/datasets/d07f4a2f-a28e-4b21-8915-c62ead50069b/compare?selectedSessions=439f1370-e399-48ac-aa94-a56ebba64104




1it [00:27, 27.95s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The provided text describes ReAct as integrati...,[page_content='Self-Reflection#\nSelf-reflecti...,None,"ReAct integrates reasoning and acting, perform...",False,True,True,False,6.236582,87560216-3df0-4519-8a08-a169c067d89b,019ce6be-25e3-7493-9a16-3ac64364c4fa
